### The Baselines

* **Random Classifier:**
* **Heuristic:**
* **Multi-Class Linear Classifier:** 

### Neural Network

* **Basic Neural Network / Multilayer Perceptron:**
* **Regularized NN:**

### Natural Language Processing:

* **Bag of Words Model:**
* **RNN Sentiment Analysis:**

### Extra

* **Attention Models:** Use attention mechanisms on the titles/descriptions
* **Some Fancy Pretrained Model maybe BERT too**
* **Just run it into an LLM and see how it does**


## Load and Process Data

In [2]:
# access parquet :D files with pandas
import pandas as pd
import polars as pl

df = pl.read_parquet('data/youtube_data.parquet')
df

title,channel_name,daily_rank,daily_movement,weekly_movement,snapshot_date,country,view_count,like_count,comment_count,description,video_id,channel_id,video_tags,kind,publish_date,langauge,like_tier
str,cat,u8,i16,i16,"datetime[μs, UTC]",cat,i64,i32,i32,str,str,str,str,cat,"datetime[μs, UTC]",cat,cat
"""Nyasha David - Tsvodi (Officia…","""Nyasha David""",1,0,15,2026-02-18 00:00:00 UTC,"""ZW""",430573,13860,1139,"""#Tsvodi #nyashadavid #mafindif…","""07pinMbPq2Q""","""UCx1LPxEtJWIpvtXTYadAdKg""","""""","""youtube#video""",2026-02-09 00:00:00 UTC,"""und""","""pretty_viral"""
"""WAR MACHINE | Official Trailer…","""Netflix""",2,0,0,2026-02-18 00:00:00 UTC,"""ZW""",12531660,191591,15336,"""During the final stage of U.S.…","""AFuE1LRxm80""","""UCWOA1ZGywLbqmigxE4Qlvuw""","""Alan Ritchson, Alan Ritchson W…","""youtube#video""",2026-02-04 00:00:00 UTC,"""en-US""","""really_viral"""
"""10 Seconds vs 1 Hour MODERN MO…","""Cash""",3,0,47,2026-02-18 00:00:00 UTC,"""ZW""",491455,6139,1656,"""Instagram: https://www.instagr…","""EbBjlznDwzk""","""UC0eLBYhxW9HC0P9PXQ73mpQ""","""Cash, Nico, Nico and Cash, Cas…","""youtube#video""",2026-02-16 00:00:00 UTC,"""en-US""","""a_little_viral"""
"""Master H - Impossible (Officia…","""Master H""",4,0,3,2026-02-18 00:00:00 UTC,"""ZW""",532388,14552,1281,"""Artist : Master H Song : Impos…","""TIG5IzusQ24""","""UC5jrU8WxzE16gPTnlqQAqew""","""""","""youtube#video""",2026-02-06 00:00:00 UTC,"""en""","""pretty_viral"""
"""“Spider-Noir” – Authentic Blac…","""Prime Video""",5,0,45,2026-02-18 00:00:00 UTC,"""ZW""",13406903,239798,10614,"""With no power comes no respons…","""HgMbkitzhEM""","""UCQJWtTnAHhEG5w4uN0udnUQ""","""spider noir official teaser, s…","""youtube#video""",2026-02-12 00:00:00 UTC,"""en""","""really_viral"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""انهيار الطبيبة هديل الجمل عند …","""شبكة قدس الاخبارية""",46,4,4,2023-10-26 00:00:00 UTC,"""AE""",1064501,8823,815,"""تابعوا #شبكة_قدس عبر المنصات ا…","""OcPejiPT07U""","""UCFFuMGTQXLitNuiU4SIe5vQ""","""فلسطين, القدس, غزة, تاريخ فلسط…","""youtube#video""",2023-10-22 15:04:53 UTC,"""ar""","""a_little_viral"""
"""بدهم ايانا نترك بيتنا !""","""وليد ونور | Waleed & Noor""",47,3,3,2023-10-26 00:00:00 UTC,"""AE""",413648,36218,2070,"""حساباتنا على مواقع التواصل الإ…","""qX9XD-5ZGIE""","""UCHywC_HXMWAM6-XvjY8gnUw""","""وليد ونور, نور غسان, وليدمقداد…","""youtube#video""",2023-10-22 16:10:45 UTC,"""ar""","""pretty_viral"""
"""""إسرائيل"" تنزف!.. هل ينهار الا…","""المخبر الاقتصادي - Mokhbir Eqt…",48,2,2,2023-10-26 00:00:00 UTC,"""AE""",2076478,197026,12930,"""لو انت شركة أو تاجر اتواصل مع …","""s7n2wzIWjtY""","""UC4kRorAXuIkyIX6vwXKaLWg""","""المخبر الاقتصادي, اخبار الاقتص…","""youtube#video""",2023-10-19 17:00:08 UTC,"""ar""","""really_viral"""


In [4]:
# Dedpulicate videos
df = df.sort("snapshot_date")
df_unique = df.unique(subset=["video_id"], keep="last")

# Sort the new unique dataframe by when the videos were published
df_unique = df_unique.sort("publish_date")

# Split by cutoff date
cutoff_date = df_unique['publish_date'].quantile(0.8, "lower")
print(f"Using publish_date {cutoff_date} as the cutoff point.\n")

train_df = df_unique.filter(pl.col("publish_date") <= cutoff_date)
test_df = df_unique.filter(pl.col("publish_date") > cutoff_date)

X_train = train_df.drop("like_tier")
y_train = train_df.get_column("like_tier")

X_test = test_df.drop("like_tier")
y_test = test_df.get_column("like_tier")

print(f"Training set size: {len(X_train)} unique videos (past)")
print(f"Testing set size:  {len(X_test)} unique videos (future)")

Using publish_date 2025-11-05 00:00:00+00:00 as the cutoff point.

Training set size: 366469 unique videos (past)
Testing set size:  90674 unique videos (future)


#### Random Classifier

In [4]:
# a random classifier that takes account in class distributions
import numpy as np
from sklearn.metrics import classification_report

# Calculate class probabilities directly from the y_train Series
counts_df = y_train.value_counts()
labels = counts_df.get_column("like_tier").to_list()
probs = (counts_df.get_column("count") / len(y_train)).to_list()

random_preds = np.random.choice(
    labels,
    size=len(y_test),
    p=probs
)

print("Random Classifier Evaluation Report:")
print(classification_report(y_test, random_preds, zero_division=0))

Random Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      0.49      0.61     73784
  pretty_viral       0.16      0.32      0.21     14234
  really_viral       0.03      0.14      0.05      2486
   super_viral       0.00      0.06      0.00       170

      accuracy                           0.46     90674
     macro avg       0.25      0.25      0.22     90674
  weighted avg       0.69      0.46      0.53     90674



### Huristic Classifiers

#### Majority Class

In [5]:
# find the most common class in the training labels
majority_class = y_train.mode()[0]

# predict that class for every test sample
heuristic_preds = [majority_class] * len(y_test)


print("Majority Class:", majority_class)
print("Huristic Classifier Evaluation Report:")
print(classification_report(y_test, heuristic_preds, zero_division=0))

Majority Class: a_little_viral
Huristic Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.81      1.00      0.90     73784
  pretty_viral       0.00      0.00      0.00     14234
  really_viral       0.00      0.00      0.00      2486
   super_viral       0.00      0.00      0.00       170

      accuracy                           0.81     90674
     macro avg       0.20      0.25      0.22     90674
  weighted avg       0.66      0.81      0.73     90674



This suggests the data is relatively balanced since we are getting an accuracy of around 25%! So there isn't one specific class that dominates over the rest which is good.

The random and majority class huristic show that the data is pretty heavily skewed to low virality so the accuracy is pretty high. However neither model has any reliability in perdicting when a given video would have mid-high virality. 

### Linear Classifiers

#### Torch model
* 50 epochs
* Numeric Features
* SGD optimizer

In [6]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# multi-class linear classifier

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
EPOCHS = 50

# numeric only first not like_count lmao
numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

# Convert everything to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

# Create Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

# Simple linear classifier
mlp_model = nn.Linear(len(numeric_features), len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
# We'll use SGD to make the baseline less powerful Adam does too much for us
optimizer = torch.optim.SGD(mlp_model.parameters(), lr=0.01)

print("Training Linear Classifier...")
for epoch in range(EPOCHS):
    mlp_model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = mlp_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

mlp_model.eval()
with torch.no_grad():
    test_outputs = mlp_model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    
    # Move predictions back to CPU for Scikit-Learn
    predicted_np = predicted.cpu().numpy()

print("\nLinear Classifier Numeric Only Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

Using device: mps
Training Linear Classifier...

Linear Classifier Numeric Only Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.88      0.99      0.93     73784
  pretty_viral       0.65      0.28      0.40     14234
  really_viral       0.72      0.32      0.45      2486
   super_viral       0.61      0.55      0.58       170

      accuracy                           0.86     90674
     macro avg       0.72      0.54      0.59     90674
  weighted avg       0.84      0.86      0.83     90674



#### Scikit LogisticRegression model
* 100 max_iter
* Numeric Features
* LBFGS solver

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.select(numeric_features).to_numpy())
X_test_scaled = scaler.transform(X_test.select(numeric_features).to_numpy())

y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

print("Training Scikit-Learn Linear Classifier...")

mlp_model = LogisticRegression(
    solver='lbfgs',      # Standard solver for multi-class
    max_iter=100,       # Gives it enough time to converge
    random_state=42      # Keeps results reproducible
)
mlp_model.fit(X_train_scaled, y_train_np)
predicted_np = mlp_model.predict(X_test_scaled)

print("\nScikit-Learn Linear Classifier Evaluation Report:")

target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_np, predicted_np, labels=target_names, zero_division=0))

Training Scikit-Learn Linear Classifier...

Scikit-Learn Linear Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.91      0.97      0.94     73784
  pretty_viral       0.67      0.45      0.54     14234
  really_viral       0.68      0.41      0.51      2486
   super_viral       0.58      0.65      0.61       170

      accuracy                           0.88     90674
     macro avg       0.71      0.62      0.65     90674
  weighted avg       0.86      0.88      0.86     90674



### Multilayer Perceptron 
* 50 Epochs
* Numerical Features
* Adam Optimizer


In [8]:
from sklearn.compose import ColumnTransformer

numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
EPOCHS = 50

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
    ])

X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)


mlp_model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(EPOCHS):
    mlp_model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = mlp_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

mlp_model.eval()
with torch.no_grad():
    test_outputs = mlp_model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension: 5 features
Training MLP Classifier...

MLP Classifier Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.94      0.94     73784
  pretty_viral       0.62      0.65      0.64     14234
  really_viral       0.64      0.53      0.58      2486
   super_viral       0.44      0.79      0.56       170

      accuracy                           0.88     90674
     macro avg       0.66      0.73      0.68     90674
  weighted avg       0.88      0.88      0.88     90674



### Multilayer Perceptron 
* 50 Epochs
* Numerical + Categorical Features
* Adam Optimizer


In [ ]:
from sklearn.preprocessing import OneHotEncoder

numeric_features = ["daily_rank", "daily_movement", "weekly_movement", "view_count", "comment_count"]
categorical_features = ["country", "langauge"] # "publish_day" - didn't exist in column
EPOCHS = 50

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension: {input_dim} features")

label_map = {
    "a_little_viral": 0, 
    "pretty_viral": 1, 
    "really_viral": 2, 
    "super_viral": 3
}

y_train_mapped = y_train.to_pandas().map(label_map).to_numpy()
y_test_mapped = y_test.to_pandas().map(label_map).to_numpy()

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_mapped, dtype=torch.long).to(device)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_mapped, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

print("Training MLP Classifier...")
for epoch in range(EPOCHS):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    predicted_np = predicted.cpu().numpy()

print("\nMLP Classifier Numeric + Categorical Evaluation Report:")
target_names = ["a_little_viral", "pretty_viral", "really_viral", "super_viral"]
print(classification_report(y_test_mapped, predicted_np, target_names=target_names, zero_division=0))

New Input Dimension: 296 features
Training MLP Classifier...

MLP Classifier Numeric + Categorical Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.82      0.78      0.80     73784
  pretty_viral       0.00      0.00      0.00     14234
  really_viral       0.03      0.22      0.05      2486
   super_viral       1.00      0.01      0.01       170

      accuracy                           0.64     90674
     macro avg       0.46      0.25      0.21     90674
  weighted avg       0.67      0.64      0.65     90674



### Natural Language Processing

#### Bag of Words + Logistic Regression


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

# combine title and description into a single text feature
X_train_pd["text"] = X_train_pd["title"] + " " + X_train_pd["description"]
X_test_pd["text"] = X_test_pd["title"] + " " + X_test_pd["description"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        # using frequency BoW
        ('bow', CountVectorizer(max_features=2000), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with BoW: {input_dim} features")

New Input Dimension with BoW: 2296 features


In [11]:
from sklearn.linear_model import LogisticRegression

# count examples per class
class_counts = np.bincount(y_train_mapped)
print("Class counts:", class_counts)

# inverse frequency weights to handle class imbalance
class_weights = {i: max(class_counts)/count for i, count in enumerate(class_counts)}
print("Class weights:", class_weights)

clf = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight=class_weights,
    n_jobs=-1
)

clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with BoW Evaluation Report:")
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

Class counts: [180600 117454  51222  17193]
Class weights: {0: np.float64(1.0), 1: np.float64(1.5376232397364074), 2: np.float64(3.525828745460935), 3: np.float64(10.50427499563776)}


/Users/sophia/Documents/GitHub/DeeplearningFinalUwU/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



Logistic Regression with BoW Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.94      0.92      0.93     73784
  pretty_viral       0.58      0.58      0.58     14234
  really_viral       0.52      0.70      0.60      2486
   super_viral       0.36      0.82      0.50       170

      accuracy                           0.86     90674
     macro avg       0.60      0.76      0.65     90674
  weighted avg       0.87      0.86      0.87     90674



/Users/sophia/Documents/GitHub/DeeplearningFinalUwU/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


#### TF-IDF + Logistic Regression

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('tfidf', TfidfVectorizer(max_features=5000, sublinear_tf=True), "text")
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_pd)
X_test_processed = preprocessor.transform(X_test_pd)

input_dim = X_train_processed.shape[1]
print(f"New Input Dimension with TF-IDF: {input_dim} features")

clf = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight=class_weights,
    n_jobs=-1
)
clf.fit(X_train_processed, y_train_mapped)
y_pred = clf.predict(X_test_processed)

print("\nLogistic Regression with TF-IDF Evaluation Report:")
print(classification_report(y_test_mapped, y_pred, target_names=target_names, zero_division=0))

New Input Dimension with TF-IDF: 5296 features


/Users/sophia/Documents/GitHub/DeeplearningFinalUwU/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



Logistic Regression with TF-IDF Evaluation Report:
                precision    recall  f1-score   support

a_little_viral       0.95      0.93      0.94     73784
  pretty_viral       0.62      0.64      0.63     14234
  really_viral       0.56      0.73      0.63      2486
   super_viral       0.46      0.89      0.60       170

      accuracy                           0.88     90674
     macro avg       0.65      0.80      0.70     90674
  weighted avg       0.88      0.88      0.88     90674



/Users/sophia/Documents/GitHub/DeeplearningFinalUwU/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


#### RNN Sentiment Analysis